# B2.2 · Threat modelling from what the estate already knows

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *Security of AI*

Builds on **[B2.1 · What building a harness means in security engineering](https://spbreed.github.io/cyber-commons/lessons/B2.1.html)**.

| | |
|---|---|
| Tools used | OWASP Threat Dragon, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Turn an architecture map into a ranked threat model, then diff it after one entry point is added.

**Why a security engineer needs it.** Threat models are written once, by hand, against a system that has since changed. The control it builds is: stage 5: derive assets, entry points and attack vectors mechanically from the synthesised map.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A threat model produced in a workshop describes the system as it was on the day of the workshop, and it is derived from the code alone — so two deployments of the same repository, one behind a private load balancer with no egress and one on the internet with a wildcard trust policy, get the same model. It is wrong about both.

> **At CyberTravels.** The threat model that said “CyberTravels answers questions” is still on file. Deriving it from the architecture on every release is what would have caught the refund endpoint appearing.

## 2 · The framework

```
   six static inputs, all already in the estate

   code analysis      what the code COULD reach     (stage 4)
   cloud policy       is it on the internet         (security groups, WAF)
   CSPM               is the bucket public TODAY
   entitlements       what the role may do
   IAM                who can become that role
   egress policy      can anything leave
          |
          v  derive, mechanically
   +---------------------------+
   | ranked threats, as data   |
   +---------------------------+
          |
        DIFFED against the last run - and the diff has to count
        ESCALATION, not only arrival, or a terraform-only pull
        request raises every score and passes the gate
```

Stage 5 is the one everybody claims to do and almost nobody re-runs.

A threat model produced in a workshop describes the system as it was on the day
of the workshop. It is stale the moment an entry point is added, and adding an
entry point is a Tuesday. So this stage does not *write* a threat model — it
**derives** one, from evidence the estate already holds, and the useful artefact
is the diff between two runs.

**STRIDE** gives six questions. Against an agentic system each has a shape a
web-application threat model does not:

| STRIDE | In an agentic system |
|---|---|
| **S**poofing | agents share a service account, so "which agent" is unanswerable |
| **T**ampering | untrusted content the agent read becomes an instruction it follows |
| **R**epudiation | the delegation chain is not on the token, so no log answers "on whose behalf" |
| **I**nformation disclosure | an over-broad tool return, or egress that permits anything |
| **D**enial of service | an unbounded loop, or a budget nobody set |
| **E**levation of privilege | a role assumable by `*`, or a scope that included refunds because it included payments |

### Derived from five inputs, not one

The architecture map says what the code *could* reach. It cannot say whether
that path is exposed, what identity walks it, or whether anything can leave at
the end of it — and those three decide whether a finding is a fire.

| Input | What only it can tell you |
|---|---|
| **architecture** | components, flows, sinks, trust levels |
| **CSPM** | that the bucket behind that sink is public *today* |
| **IAM** | who can assume the role, and whether MFA is required |
| **network** | is it internet-facing, and can anything leave |
| **entitlements** | what the identity may do once it is through |

Read only the first and you produce a model that is identical for two
deployments of the same repository — one behind a private load balancer with no
egress, one on the internet with a wildcard trust policy. It is wrong about
both.

This lesson runs the `threat-model-stride` skill. You do not write the code;
you read the procedure, execute it, and read what it produced.

## 3 · The skill

This is `skills/appsec/threat-model-stride/SKILL.md`, verbatim. The frontmatter is what routes a request to it; the body is the procedure a model follows.

### The skill — [`skills/appsec/threat-model-stride/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/threat-model-stride/SKILL.md)

```yaml
name: threat-model-stride
description: >-
  Build a STRIDE threat model for an agentic system from the evidence an estate
  already holds — source code, CSPM findings, IAM policies, network policies and
  the tool surface — and emit a ranked threat table plus a trust-boundary
  diagram. Use when asked what could go wrong with an architecture, to
  threat-model an agent, a pipeline or a service, when a threat model needs
  regenerating after an architecture change, or when someone asks which STRIDE
  categories a design actually exposes.
allowed-tools: Read, Grep, Glob, Bash
```

# STRIDE threat modelling for an agentic system

A threat model written in a workshop describes the system as it was on the day
of the workshop. This one is **derived**: every threat traces to a line of
evidence that already exists somewhere in the estate, so regenerating it is
cheap and the useful artefact is the **diff between two runs**.

STRIDE gives six categories. Against an agentic system each one has a specific
shape that a web-application threat model does not:

| STRIDE | In an agentic system |
|---|---|
| **S**poofing | An agent calls downstream as a shared service account, so "which agent" is unanswerable |
| **T**ampering | Untrusted content the agent read becomes an instruction it follows |
| **R**epudiation | The delegation chain is not on the token, so no log answers "on whose behalf" |
| **I**nformation disclosure | An over-broad tool return, or egress that permits anything |
| **D**enial of service | An unbounded loop, or a budget nobody set |
| **E**levation of privilege | A role assumable by `*`, or a scope that includes refunds because it included payments |

## When to use this

Threat-modelling an agent, an MCP server, or a review pipeline; re-running a
model after an architecture change; or answering "which STRIDE categories does
this design actually expose" with something better than an opinion.

## Inputs it expects

Five evidence sources. Any one alone produces a model that is wrong in a
predictable direction — code alone cannot tell you whether a path is exposed,
and CSPM alone cannot tell you what reaches it.

| Input | What only it can tell you |
|---|---|
| `architecture` | components, flows, sinks, trust levels — what *could* be reached |
| `cspm` | live posture findings: what is public *today* |
| `iam` | which roles exist, who may assume them, whether MFA is required |
| `network` | ingress exposure and egress policy — can anything leave |
| `entitlements` | what the running identity may do once it is through |

## Procedure

**1 — Load the five inputs.** Refuse to proceed on fewer. A model built on
`architecture` alone should say so in its output rather than silently scoring as
though the estate were hardened.

**2 — Walk each entry point to each sink.** For every reachable pair, ask the
six STRIDE questions. A category that has no evidence behind it is not a threat;
do not invent one to fill the row.

**3 — Score from evidence, not from feeling.** Base severity comes from the
asset. Then adjust *only* where an input says so: internet-facing, no WAF, a
live CSPM finding on the resource, a role that holds write, a trust policy with
a wildcard, egress open. Record which adjustment fired — the reasons are the
part a reviewer argues with.

**4 — Emit the trust-boundary diagram.** Nodes are components, edges are flows,
and an edge crossing from a lower trust level to a higher one is a boundary.
Boundaries are where findings live; render them differently from ordinary edges.

**5 — Diff against the previous run.** Report new threats *and* escalated ones.
A pull request that changes only infrastructure introduces no new threat and
raises every existing score, so a gate that counts arrivals alone waves it
through.

## Example

**Input** — the fixture committed at the top of [`scripts/threat_model.py`](scripts/threat_model.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
id      entry           sink           score  why
--------------------------------------------------------------------------------------------
T-10  I upload_voucher  store             11  a live CSPM finding on the resource this path reaches
T-01  S get_booking     load_booking      11  the running role is assumable by *
T-05  D get_booking     load_booking      10  internet-facing with no WAF in front of it
T-06  E get_booking     load_booking      10  the identity holds write, not just read
T-03  R get_booking     load_booking      10  no MFA on the assumable role, so the actor is not established
T-04  I get_booking     load_booking       9  no open CSPM finding on the resource
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "inputs_present": ["architecture", "cspm", "iam", "network", "entitlements"],
  "threats": [
    {"id": "T-01", "stride": "S|T|R|I|D|E", "entry": "str", "sink": "str",
     "asset": "str", "score": 0, "reasons": ["internet-facing", "..."],
     "evidence": {"source": "cspm|iam|network|entitlements|architecture",
                  "detail": "str"}}
  ],
  "boundaries": [{"from": "str", "to": "str", "trust": "0->2"}],
  "diagram": "mermaid or svg source",
  "diff": {"new": ["T-07"], "escalated": [{"id": "T-01", "from": 13, "to": 17}]}
}
```

## Failure modes

- **Modelling the code and calling it the system.** The same repository behind
  a private load balancer with default-deny egress and on the internet with a
  wildcard trust policy is two different threat models. Read all five inputs.
- **Counting new threats only.** Escalation is the signal a terraform-only
  change produces, and it is the majority of how an estate gets worse.
- **One row per STRIDE letter.** Six categories is a checklist, not a quota. An
  agent with no state has no meaningful tampering row.
- **Scoring without recording why.** A score nobody can argue with is a score
  nobody will act on.

## 4 · Its script

The deterministic half of the skill — the part that has to give the same answer twice so two runs can be diffed. Embedded from `skills/appsec/threat-model-stride/scripts/`.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/threat-model-stride/scripts/threat_model.py
SCRIPT = "skills/appsec/threat-model-stride/scripts/threat_model.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 5 · Execute it against CyberTravels

Five synthetic inputs, standing in for what a real estate already holds.

## 6 · The diagram it emits

Mermaid, so it renders here and on the lesson page without a library. Double arrows are trust-boundary crossings — the edges every finding turned out to live on.

## 7 · The same code, a hardened estate

Not one line of CyberTravels' source changes. Only the four evidence inputs around it do.

## What you just proved

The skill loads with its routing description and procedure, then derives twelve threats across all six STRIDE categories from five synthetic inputs, each carrying the evidence line that set its score. It emits a mermaid diagram marking the two trust-boundary crossings. Re-running against a hardened estate — same code, four different evidence inputs — keeps every row and drops the maximum severity from 11 to 1.

## Your turn

Point the skill at one of your own services. The work is not the model, it is collecting the five inputs: if any of them is "in somebody's head", that is the input your threat model is currently guessing at, and the guess is always the optimistic one.

---

**Next → [B2.3 · Vulnerability auditing — deterministic Semgrep, then the model pass](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*